## 📌 Prerequisites for Demo

Before running this demo, please ensure the following setup steps are completed:

---

#### **1 Start the Typesense server**

- Install Docker Desktop

We use **Typesense** for document indexing and retrieval.

```bash
docker run -p 8108:8108 \
  -v/tmp/typesense-data:/data \
  typesense/typesense:0.24.1 \
  --data-dir /data \
  --api-key=xyz \
  --enable-cors
```

#### **2 Create virtual environment and insatll dependecies**

```bash
uv venv rag

rag\Scripts\activate #(for windows)

uv pip install -e .
```
- In the top right corner select kernel as rag

#### **3 Set the .env file and run the Rag api**

- Get the gemini api key and set this in you .env(change the typesense variables if you have changed in the above docker run)
GEMINI_API_KEY=
TYPESENSE_API_KEY=xyz  
TYPESENSE_HOST=localhost
TYPESENSE_PORT=8108
TYPESENSE_PROTOCOL=http
```bash
uvicorn main:api --reload --port 8000
```




In [79]:
## Doing the imports and setting up the URLs
import requests
from pathlib import Path
import time
import json

BASE_URL = "http://localhost:8000"  # FastAPI URL
UPLOAD_URL = f"{BASE_URL}/upload"
ASK_URL = f"{BASE_URL}/ask"

Adding Test User

In [80]:
USER_ID = "test_user"  # Your test user ID

Uploading the document

**Note - i have added a sample document under test_files folder

- After the upload is done you will be able to see the files at user_uploads\{user_name}\processed

In [81]:
# --- Prepare files from sarthak folder ---
folder_path = Path("sarthak")  # Adjust path if needed
files_to_upload = [
    ("files", (f.name, open(f, "rb"), "application/octet-stream"))
    for f in folder_path.iterdir() if f.is_file()
]

# --- Upload documents ---
upload_resp = requests.post(UPLOAD_URL, data={"user_id": USER_ID}, files=files_to_upload)
print("Upload response:", upload_resp.status_code, upload_resp.json())


Upload response: 200 {'message': 'Files uploaded, processed, and moved successfully', 'files_processed': ['conference_paper.pdf']}


Hitting ASk to get the response

- we are first giving a simple questions and checking the response 
- I have tried to implement a logic where user can see from which document and page number this information was fetched ( can be combined with UI for exact citations)

In [83]:
query = "Who is the GPIO?"
payload = {"user_id": USER_ID, "query": query}

ask_resp = requests.post(ASK_URL, json=payload)
print(ask_resp.status_code)
ask_resp = ask_resp.json()
print(json.dumps(ask_resp, indent=2))

200
{
  "answer": {
    "content": "GPIO refers to pins on a Raspberry Pi that allow for direct communication with sensors like ultrasonic sensors and cameras. These pins can function as both input and output, and can also be used for power input. A software library called RPi.GPIO is used to get readings from sensors connected to these pins.",
    "citations": {
      "user_uploads\\test_user\\processed\\conference_paper.pdf": [
        4,
        3,
        2
      ]
    }
  }
}


Cache Demo

-- we hit the query twice and get the responce quickly

In [84]:
# --- Ask question (first time, should run retrieval+LLM) ---
query = "Who is the Rpi?"
payload = {"user_id": USER_ID, "query": query}

start = time.perf_counter()
ask_resp = requests.post(ASK_URL, json=payload)
elapsed = time.perf_counter() - start
print(f"Ask (1st time) took {elapsed:.2f}s:", ask_resp.status_code)
print(ask_resp.json())

# --- Ask question again (should hit cache) ---
start = time.perf_counter()
ask_resp2 = requests.post(ASK_URL, json=payload)
elapsed2 = time.perf_counter() - start
print(f"Ask (2nd time, cache) took {elapsed2:.2f}s:", ask_resp2.status_code, ask_resp2.json())
print(ask_resp2.json())

Ask (1st time) took 16.26s: 200
{'answer': {'content': 'The Raspberry Pi (Rpi) is a chip, the size of a credit card, that can function as a computer running the Raspberry Pi OS, which is similar to a Linux environment. Its primary advantage is its GPIO pins, which facilitate direct communication with sensors. It can be connected to peripherals like a monitor and keyboard via wired or wireless connections. Additionally, a low-cost HD camera module can be directly connected to the Pi device.', 'citations': {'user_uploads\\test_user\\processed\\conference_paper.pdf': [2]}}}
Ask (2nd time, cache) took 2.06s: 200 {'answer': {'content': 'The Raspberry Pi (Rpi) is a chip, the size of a credit card, that can function as a computer running the Raspberry Pi OS, which is similar to a Linux environment. Its primary advantage is its GPIO pins, which facilitate direct communication with sensors. It can be connected to peripherals like a monitor and keyboard via wired or wireless connections. Additio

Ask the query about conersation history

In [85]:
query = "what is the last question?"
payload = {"user_id": USER_ID, "query": query}

start = time.perf_counter()
ask_resp = requests.post(ASK_URL, json=payload)
elapsed = time.perf_counter() - start
print(f"Ask (1st time) took {elapsed:.2f}s:", ask_resp.status_code)
print(ask_resp.json())

Ask (1st time) took 7.74s: 200
{'answer': {'content': 'The last question was: Who is the Rpi?', 'citations': {}}}


Creating another user

In [87]:
USER_ID = "another user"

If the user has no documents it will not answer

In [88]:
query = "Who is the GPIO?"
payload = {"user_id": USER_ID, "query": query}

ask_resp = requests.post(ASK_URL, json=payload)
print(ask_resp.status_code)
ask_resp = ask_resp.json()
print(json.dumps(ask_resp, indent=2))

200
{
  "answer": {
    "content": null,
    "citations": {}
  }
}


Printing Histry and Cache

In [90]:
from memory_management_and_caching.cache_manager import CacheManager
from memory_management_and_caching.conversation_manager import ConversationManager
from db.db_create import db_client
import json

# --- Cache listing helper ---
def list_cache(cache_manager, user_id: str = None, limit: int = 10):
    """List recent cache entries in latest-first order."""
    try:
        filter_str = f"user_id:={user_id}" if user_id else ""
        search = cache_manager.client.collections[cache_manager.collection_name].documents.search({
            "q": "*",
            "query_by": "query",
            "filter_by": filter_str,
            "per_page": limit,
        })
        return [
            {
                "user_id": hit["document"]["user_id"],
                "query": hit["document"]["query"],
                "answer": json.loads(hit["document"]["answer_json"])
            }
            for hit in search.get("hits", [])
        ]
    except Exception as e:
        print(f"Error listing cache: {e}")
        return []

# --- Conversation history listing helper ---
def list_conversation_history(conv_manager, user_id: str, limit: int = 10):
    """List conversation history for a user, latest-first."""
    try:
        history = conv_manager.get_conversation_history(user_id, last_n=limit)
        return list(reversed(history))  # reverse to get latest first
    except Exception as e:
        print(f"Error listing conversation history: {e}")
        return []

# --- Create client & managers ---
client = db_client.get_client()
cache_manager = CacheManager(client)
conv_manager = ConversationManager(client)

Full cache

In [91]:
# --- View all cache entries ---
print("---- All Cache Entries ----")
all_entries = list_cache(cache_manager, limit=5)
for entry in all_entries:
    print(json.dumps(entry, indent=2))


---- All Cache Entries ----
{
  "user_id": "test_user",
  "query": "what is the last question?",
  "answer": {
    "content": "The last question was: Who is the Rpi?",
    "citations": {}
  }
}
{
  "user_id": "test_user",
  "query": "who is the rpi?",
  "answer": {
    "content": "The Raspberry Pi (Rpi) is a chip, the size of a credit card, that can function as a computer running the Raspberry Pi OS, which is similar to a Linux environment. Its primary advantage is its GPIO pins, which facilitate direct communication with sensors. It can be connected to peripherals like a monitor and keyboard via wired or wireless connections. Additionally, a low-cost HD camera module can be directly connected to the Pi device.",
    "citations": {
      "user_uploads\\test_user\\processed\\conference_paper.pdf": [
        2
      ]
    }
  }
}
{
  "user_id": "test_user",
  "query": "who is the gpio?",
  "answer": {
    "content": "GPIO refers to pins on a Raspberry Pi that allow for direct communicati

Cache of single user

In [92]:
# --- View cache for a specific user ---
print("\n---- Cache for user ----")
user_entries = list_cache(cache_manager, user_id="test_user", limit=5)
for entry in user_entries:
    print(json.dumps(entry, indent=2))




---- Cache for user ----
{
  "user_id": "test_user",
  "query": "what is the last question?",
  "answer": {
    "content": "The last question was: Who is the Rpi?",
    "citations": {}
  }
}
{
  "user_id": "test_user",
  "query": "who is the rpi?",
  "answer": {
    "content": "The Raspberry Pi (Rpi) is a chip, the size of a credit card, that can function as a computer running the Raspberry Pi OS, which is similar to a Linux environment. Its primary advantage is its GPIO pins, which facilitate direct communication with sensors. It can be connected to peripherals like a monitor and keyboard via wired or wireless connections. Additionally, a low-cost HD camera module can be directly connected to the Pi device.",
    "citations": {
      "user_uploads\\test_user\\processed\\conference_paper.pdf": [
        2
      ]
    }
  }
}
{
  "user_id": "test_user",
  "query": "who is the gpio?",
  "answer": {
    "content": "GPIO refers to pins on a Raspberry Pi that allow for direct communication

View Conversation history

In [93]:
# --- View conversation history for a user ---
print("\n---- Conversation History for user ----")
history_entries = list_conversation_history(conv_manager, user_id="test_user", limit=5)
for h in history_entries:
    print(json.dumps(h, indent=2))


---- Conversation History for user ----
{
  "query": "what is the last question?",
  "answer": {
    "content": "The last question was: Who is the Rpi?",
    "citations": {}
  }
}
{
  "query": "Who is the Rpi?",
  "answer": {
    "content": "The Raspberry Pi (Rpi) is a chip, the size of a credit card, that can function as a computer running the Raspberry Pi OS, which is similar to a Linux environment. Its primary advantage is its GPIO pins, which facilitate direct communication with sensors. It can be connected to peripherals like a monitor and keyboard via wired or wireless connections. Additionally, a low-cost HD camera module can be directly connected to the Pi device.",
    "citations": {
      "user_uploads\\test_user\\processed\\conference_paper.pdf": [
        2
      ]
    }
  }
}
{
  "query": "Who is the GPIO?",
  "answer": {
    "content": "GPIO refers to pins on a Raspberry Pi that allow for direct communication with sensors like ultrasonic sensors and cameras. These pins c

IF you want to reset

- Run the below code restart the fast API application 

In [ ]:

from db.db_create import db_client

client = db_client.get_client()


try:
    client.collections["qa_cache"].delete()
    print("✅ Cache cleared: qa_cache collection deleted.")
except Exception as e:
    print("Cache clear error:", e)

try:
    client.collections["documents"].delete()
    print("✅ Cache cleared: qa_cache collection deleted.")
except Exception as e:
    print("Cache clear error:", e)

try:
    client.collections["conversation_memory"].delete()
    print("✅ Cache cleared: qa_cache collection deleted.")
except Exception as e:
    print("Cache clear error:", e)




✅ Cache cleared: qa_cache collection deleted.
✅ Cache cleared: qa_cache collection deleted.
✅ Cache cleared: qa_cache collection deleted.
